# LaCAM run variability

`movingai_final.csv` contains four genuine LaCAM runs per instance, one for each seed from 0 to 3. The first table shows all four runs of one instance; the final section summarizes variability across ten randomly selected instances solved by every seed.

In [4]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "datasets" / "movingai_final.csv").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Run this notebook from inside the repository")
    ROOT = ROOT.parent

columns = [
    "instance_id", "lacam_seed", "lacam_solved", "lacam_runtime_seconds",
    "lacam_initial_soc", "lacam_initial_sod_over_lb", "lns_neighborhood_size",
]
data = (
    pd.read_csv(ROOT / "datasets" / "movingai_final.csv", usecols=columns)
    .query("lns_neighborhood_size == 4")
    .drop_duplicates(["instance_id", "lacam_seed"])
)

## Controlled seed comparison

Ten reproducibly sampled instances from `movingai_final`, each solved with all four recorded LaCAM seeds.

In [5]:
eligible = (
    data.groupby("instance_id")
    .agg(seed_count=("lacam_seed", "nunique"), solved_count=("lacam_solved", "sum"))
    .query("seed_count == 4 and solved_count == 4")
)
sample = eligible.sample(10, random_state=42).index.sort_values().tolist()
seed_runs = (
    data[data["instance_id"].isin(sample)][
        ["instance_id", "lacam_seed", "lacam_runtime_seconds", "lacam_initial_soc"]
    ]
    .rename(columns={
        "lacam_seed": "seed",
        "lacam_runtime_seconds": "runtime (s)",
        "lacam_initial_soc": "SOC",
    })
    .sort_values(["instance_id", "seed"])
)
with pd.option_context("display.max_rows", None):
    display(seed_runs.round(4))

,instance_id,seed,runtime (s),SOC
3060,den520d-random-11-density-b4-n5322,0,24.412,2148847.0
22860,den520d-random-11-density-b4-n5322,1,24.305,2162386.0
42660,den520d-random-11-density-b4-n5322,2,24.503,2163644.0
62460,den520d-random-11-density-b4-n5322,3,24.862,2157487.0
4828,empty-48-48-random-10-density-b2-n235,0,0.038,7489.0
24628,empty-48-48-random-10-density-b2-n235,1,0.044,7489.0
44428,empty-48-48-random-10-density-b2-n235,2,0.052,7489.0
64228,empty-48-48-random-10-density-b2-n235,3,0.037,7489.0
6844,ht_mansion_n-random-19-density-b2-n954,0,3.529,194798.0
26644,ht_mansion_n-random-19-density-b2-n954,1,3.525,193974.0


In [6]:
run_variability = seed_runs.groupby("instance_id").agg(
    runtime_mean_s=("runtime (s)", "mean"),
    runtime_std_s=("runtime (s)", "std"),
    soc_mean=("SOC", "mean"),
    soc_std=("SOC", "std"),
)
display(run_variability.round(3))

,runtime_mean_s,runtime_std_s,soc_mean,soc_std
instance_id,,,,
den520d-random-11-density-b4-n5322,24.520,0.242,2158091.00,6710.668
empty-48-48-random-10-density-b2-n235,0.043,0.007,7489.00,0.000
ht_mansion_n-random-19-density-b2-n954,3.494,0.305,193970.75,3003.183
maze-32-32-4-random-1-density-b6-n230,0.116,0.013,24936.00,260.633
maze-32-32-4-random-20-density-b6-n254,0.315,0.267,41147.25,4487.894
random-32-32-10-random-5-density-b2-n63,0.010,0.003,1461.00,0.000
room-32-32-4-random-10-density-b3-n107,0.025,0.004,4334.25,46.082
room-32-32-4-random-23-density-b1-n37,0.007,0.003,1053.50,3.416
warehouse-10-20-10-2-1-random-21-density-b1-n170,0.754,1.266,14796.50,157.500
